# 05 · Nitrate Adsorption Kinetics & Isotherm Modeling

## Objective
Evaluate adsorption kinetics using three classical isotherm models:
- **Langmuir:** Monolayer adsorption with finite saturation capacity
- **Freundlich:** Multilayer adsorption on heterogeneous surfaces
- **Temkin:** Intermediate model accounting for adsorbent-adsorbate interactions

## Goal
Determine which model best fits the experimental data and extract kinetic parameters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from utils.analysis_tools import determine_best_isotherm, calculate_r_squared

# Experimental equilibrium data
Ce = np.array([0.82, 1.54, 2.91, 4.63, 6.78, 9.12, 12.35, 16.20, 20.87])  # Equilibrium concentration (mg/L)
qe = np.array([0.13, 0.23, 0.40, 0.57, 0.74, 0.89, 1.04, 1.19, 1.31])     # Adsorbed amount (mg/g)

df = pd.DataFrame({'Ce': Ce, 'qe': qe})
print(df)
print(f"\nDataset: {len(df)} equilibrium points")

In [ ]:
# Fit all three isotherms and identify best model
results, best_model = determine_best_isotherm(Ce, qe)

print("="*70)
print("ISOTHERM FITTING RESULTS")
print("="*70)

for model_name, params in results.items():
    print(f"\n{model_name.upper()}:")
    print(f"  R² = {params['R2']:.4f}")
    for key, val in params.items():
        if key != 'R2':
            print(f"  {key} = {val:.4f}")

print(f"\n{'='*70}")
print(f"✓ BEST FIT MODEL: {best_model.upper()}")
print(f"  R² = {results[best_model]['R2']:.4f}")
print("="*70)

In [ ]:
# Generate predictions for plotting
Ce_range = np.linspace(Ce.min(), Ce.max(), 100)

# Langmuir: qe = (qmax * b * Ce) / (1 + b * Ce)
qmax = results['Langmuir']['qmax']
b = results['Langmuir']['b']
qe_lang = (qmax * b * Ce_range) / (1 + b * Ce_range)

# Freundlich: qe = Kf * Ce^(1/n)
Kf = results['Freundlich']['Kf']
n = results['Freundlich']['n']
qe_freund = Kf * (Ce_range ** (1/n))

# Temkin: qe = (ln(A_T * Ce)) / B_tem
A_T = results['Temkin']['A_T']
B_tem = results['Temkin']['B_tem']
qe_temkin = (np.log(A_T * Ce_range)) / B_tem

print("Prediction curves generated for plotting...")

In [ ]:
# Plot all three isotherm models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Langmuir
axes[0].scatter(Ce, qe, color='#38BDF8', s=80, zorder=4, label='Experimental')
axes[0].plot(Ce_range, qe_lang, color='#10B981', linestyle='--', linewidth=2, label='Fit')
axes[0].set_xlabel('Ce (mg/L)', fontweight='bold')
axes[0].set_ylabel('qe (mg/g)', fontweight='bold')
axes[0].set_title(f"Langmuir (R² = {results['Langmuir']['R2']:.4f})", fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Freundlich (semi-log)
axes[1].scatter(Ce, qe, color='#F59E0B', s=80, zorder=4, label='Experimental')
axes[1].plot(Ce_range, qe_freund, color='#EF4444', linestyle='--', linewidth=2, label='Fit')
axes[1].set_xlabel('Ce (mg/L)', fontweight='bold')
axes[1].set_ylabel('qe (mg/g)', fontweight='bold')
axes[1].set_title(f"Freundlich (R² = {results['Freundlich']['R2']:.4f})", fontweight='bold')
axes[1].set_xscale('log')
axes[1].grid(True, alpha=0.3, which='both')
axes[1].legend()

# Temkin
axes[2].scatter(Ce, qe, color='#8B5CF6', s=80, zorder=4, label='Experimental')
axes[2].plot(Ce_range, qe_temkin, color='#06B6D4', linestyle='--', linewidth=2, label='Fit')
axes[2].set_xlabel('Ce (mg/L)', fontweight='bold')
axes[2].set_ylabel('qe (mg/g)', fontweight='bold')
axes[2].set_title(f"Temkin (R² = {results['Temkin']['R2']:.4f})", fontweight='bold')
axes[2].set_xscale('log')
axes[2].grid(True, alpha=0.3, which='both')
axes[2].legend()

plt.tight_layout()
plt.savefig('isotherm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved as 'isotherm_comparison.png'")

## Model Interpretation

### Langmuir Model
- **Equation:** $q_e = \frac{q_{max} \cdot b \cdot C_e}{1 + b \cdot C_e}$
- **Assumptions:** Monolayer coverage, uniform binding sites, no lateral interactions
- **Parameters:**
  - $q_{max}$: Saturation capacity (mg/g)
  - $b$: Binding affinity (L/mg)

### Freundlich Model
- **Equation:** $q_e = K_f \cdot C_e^{1/n}$
- **Assumptions:** Multilayer adsorption, heterogeneous surface
- **Parameters:**
  - $K_f$: Adsorption capacity
  - $n$: Intensity of adsorption (n > 1 = favorable)

### Temkin Model
- **Equation:** $q_e = \frac{\ln(A_T \cdot C_e)}{B}$
- **Assumptions:** Linear decrease in adsorption heat with surface coverage
- **Parameters:**
  - $A_T$: Equilibrium binding constant
  - $B_{tem}$: Related to adsorption heat

In [ ]:
# Export results to CSV for documentation
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'R_Squared': [results[m]['R2'] for m in results.keys()],
    'Parameters': [
        f"qmax={results['Langmuir']['qmax']:.4f}, b={results['Langmuir']['b']:.4f}",
        f"Kf={results['Freundlich']['Kf']:.4f}, n={results['Freundlich']['n']:.4f}",
        f"A_T={results['Temkin']['A_T']:.4f}, B={results['Temkin']['B_tem']:.4f}"
    ]
})

summary_df.to_csv('isotherm_results.csv', index=False)
print("\nResults exported to 'isotherm_results.csv'")
print(summary_df)